# Comprehensive Neurophysiological Analysis Pipeline: Theory, Functions & Synthetic Exemplars
This interactive notebook contains all the mathematical definitions, standalone executable functions, and synthetic validation tests (high vs. low score exemplars) for all the single-cell and population metrics developed in our research pipeline.

## 1. Placeness Score (Spatial Information Z-Score)
### Theory
Quantifies how robustly a neuron fires at specific spatial bins along a linear track. Based on Shannon's Mutual Information adapted for spatial coding (Skaggs et al., 1993):
$$\text{SI} = \sum_i P_i \frac{\lambda_i}{\bar{\lambda}} \log_2 \left(\frac{\lambda_i}{\bar{\lambda}}\right)$$
To establish statistical significance, temporal rolling/circular shuffles are performed to compute a Z-score relative to the null distribution: $Z_{\text{SI}} = \frac{\text{SI}_{\text{actual}} - \mu_{\text{shuf}}}{\sigma_{\text{shuf}}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy import stats, signal

def compute_synthetic_placeness(n_bins=50, n_shuffles=30):
    """
    Generates synthetic high-placeness (place cell) and low-placeness (non-place cell) traces
    and computes their Spatial Information Z-scores.
    """
    np.random.seed(42)
    n_pts = 3000
    pos = np.linspace(0, 100, n_pts)
    
    # High score: focused firing field around position 50
    trace_high = np.exp(-((pos - 50.0)**2) / 50.0) * (1.0 + 0.2 * np.random.randn(n_pts))
    trace_high = np.clip(trace_high, 0, None)
    
    # Low score: uniform random noise
    trace_low = np.random.rand(n_pts)
    
    def get_si(p, tr):
        bin_edges = np.linspace(np.min(p), np.max(p), n_bins + 1)
        bin_idx = np.clip(np.digitize(p, bin_edges[1:-1]), 0, n_bins - 1)
        occ = np.bincount(bin_idx, minlength=n_bins).astype(float)
        act = np.bincount(bin_idx, weights=tr, minlength=n_bins).astype(float)
        tune = np.zeros(n_bins)
        valid = occ > 0
        tune[valid] = act[valid] / occ[valid]
        prob = occ / np.sum(occ) if np.sum(occ) > 0 else np.ones(n_bins) / n_bins
        mean_rate = np.sum(prob * tune)
        if mean_rate <= 1e-12:
            return 0.0
        ratio = tune / mean_rate
        ratio[ratio <= 0] = 1e-12
        return float(np.sum(prob * ratio * np.log2(ratio)))
        
    si_high = get_si(pos, trace_high)
    si_low = get_si(pos, trace_low)
    return si_high, si_low

sh_high, sh_low = compute_synthetic_placeness()
print(f'Synthetic Placeness Test -> High Place Score SI: {sh_high:.3f} | Low Place Score SI: {sh_low:.3f}')

## 2. Entorhinal Gridness Score (Grid Cells)
### Theory
Grid cells in the medial entorhinal cortex (MEC) exhibit periodic hexagonal firing fields in open fields (Hafting et al., 2005). The **Gridness Score** evaluates the 6-fold rotational symmetry of the 2D spatial autocorrelation map by comparing correlations at angles of 60° and 120° against 30°, 90°, and 150°.
$$\text{Gridness} = \min(\text{Corr}_{60^\circ, 120^\circ}) - \max(\text{Corr}_{30^\circ, 90^\circ, 150^\circ})$$

In [ ]:
from scipy.ndimage import rotate

def compute_gridness_score(rate_map: np.ndarray) -> float:
    """
    Computes the 6-fold rotational symmetry gridness score from a 2D spatial rate map.
    """
    mean_centered = rate_map - np.nanmean(rate_map)
    mean_centered = np.nan_to_num(mean_centered)
    acorr = signal.correlate2d(mean_centered, mean_centered, mode='full')
    acorr /= (np.max(np.abs(acorr)) + 1e-12)
    
    h, w = acorr.shape
    y, x = np.ogrid[:h, :w]
    center_y, center_x = h // 2, w // 2
    r = np.sqrt((x - center_x)**2 + (y - center_y)**2)
    mask = (r >= 5) & (r <= min(h, w) // 2)
    
    corrs = []
    for angle in [30, 60, 90, 120, 150]:
        rotated = rotate(acorr, angle, reshape=False, mode='constant', cval=0.0)
        v1 = acorr[mask]
        v2 = rotated[mask]
        valid = np.isfinite(v1) & np.isfinite(v2)
        if np.sum(valid) > 10:
            r_val, _ = stats.pearsonr(v1[valid], v2[valid])
            corrs.append((angle, r_val if np.isfinite(r_val) else 0.0))
        else:
            corrs.append((angle, 0.0))
            
    corr_dict = dict(corrs)
    min_hex = min(corr_dict.get(60, 0), corr_dict.get(120, 0))
    max_non_hex = max(corr_dict.get(30, 0), corr_dict.get(90, 0), corr_dict.get(150, 0))
    return float(min_hex - max_non_hex)

# Synthetic Grid pattern vs Random noise pattern
x_grid, y_grid = np.meshgrid(np.linspace(0, 10, 50), np.linspace(0, 10, 50))
synthetic_grid_map = np.sin(x_grid) + np.sin(0.5 * x_grid + 0.866 * y_grid) + np.sin(-0.5 * x_grid + 0.866 * y_grid)
synthetic_noise_map = np.random.rand(50, 50)

print('Synthetic Grid Cell Score:', compute_gridness_score(synthetic_grid_map))
print('Synthetic Non-Grid Noise Score:', compute_gridness_score(synthetic_noise_map))

## 3. Directionality Score (DI) & Left/Right Correlation
### Theory
Measures whether a neuron fires preferentially during outward vs. inward traversals on a linear track. 
$$\text{DI} = \frac{|\bar{R}_{\text{pos}} - \bar{R}_{\text{neg}}|}{\bar{R}_{\text{pos}} + \bar{R}_{\text{neg}} K_e}$$
Left/Right correlation evaluates the Pearson correlation coefficient between the spatial tuning curves of opposing movement directions.

In [ ]:
def compute_directionality_synthetic(mov_dir, pos, trace):
    mask_pos = (mov_dir > 0) & np.isfinite(trace)
    mask_neg = (mov_dir < 0) & np.isfinite(trace)
    mean_pos = float(np.mean(trace[mask_pos])) if np.sum(mask_pos) > 5 else 0.0
    mean_neg = float(np.mean(trace[mask_neg])) if np.sum(mask_neg) > 5 else 0.0
    denom = mean_pos + mean_neg
    di_score = float(abs(mean_pos - mean_neg) / denom) if denom > 1e-12 else 0.0
    return di_score

n = 1000
d_ex = np.random.choice([-1, 1], size=n)
p_ex = np.random.rand(n)
tr_directional = np.where(d_ex > 0, 2.0 * p_ex, 0.1 * p_ex)
print('Synthetic Directionality Score (High Directional Cell):', compute_directionality_synthetic(d_ex, p_ex, tr_directional))

## 4. Temporal Drift Score
### Theory
Quantifies non-stationarity or progressive changes in baseline/firing rates across the recording session using the absolute Spearman rank correlation coefficient between session time and neural trace: $|\rho_{\text{spearman}}(\text{time}, \text{trace})|$.

In [ ]:
def compute_drift_synthetic():
    t = np.linspace(0, 300, 1500)
    tr_drift = 0.01 * t + 0.1 * np.random.randn(1500)
    rho, _ = stats.spearmanr(t, tr_drift)
    return float(abs(rho))

print('Synthetic Temporal Drift Score:', compute_drift_synthetic())

## 5. Dual-Scale Oscillation Metrics (Fast Score & First Packet Slow Modulation Score)
### Theory
Captures temporal organization at two distinct scales via autocorrelation analysis:
1. **Fast Score ($T_{	ext{fast}}$)**: Prominence of the first autocorrelation peak within 0.5s - 30.0s.
2. **Slow Score / First Packet ($T_{	ext{slow}}$)**: Prominence of the first re-correlation envelope peak within the 30s - 100s window (capturing amplitude modulation where fast envelopes follow slow packets).

In [ ]:
def compute_dual_score_synthetic(trace, fs=20.0):
    x = trace - np.mean(trace)
    var = np.var(trace)
    if var <= 1e-12:
        return 0.0, 0.0
    autocorr = signal.correlate(x, x, mode='full')[len(trace) - 1:]
    autocorr = autocorr / (len(trace) * var)
    lags_s = np.arange(len(autocorr)) / fs
    smooth_acorr = gaussian_filter1d(autocorr, sigma=max(1, int(fs * 0.5)))
    
    fast_mask = (lags_s >= 0.5) & (lags_s <= 30.0)
    fast_score = 0.0
    if np.any(fast_mask):
        peaks, props = signal.find_peaks(smooth_acorr[fast_mask], prominence=0.005)
        if len(peaks) > 0:
            fast_score = float(np.max(props['prominences']))
            
    envelope = np.abs(signal.hilbert(smooth_acorr))
    slow_mask = (lags_s >= 30.0) & (lags_s <= 100.0)
    slow_score = 0.0
    if np.any(slow_mask):
        peaks_s, props_s = signal.find_peaks(envelope[slow_mask], prominence=0.01)
        if len(peaks_s) > 0:
            slow_score = float(np.max(props_s['prominences']))
            
    return fast_score, slow_score

t_vec = np.linspace(0, 300, 6000)
synthetic_osc = np.sin(2 * np.pi * 0.2 * t_vec) * (1.0 + 0.5 * np.cos(2 * np.pi * 0.02 * t_vec))
f_score, s_score = compute_dual_score_synthetic(synthetic_osc)
print(f'Synthetic Dual-Score Test -> Fast Oscillation Score: {f_score:.3f} | Slow Modulation Score: {s_score:.3f}')

## 6. Behavioral Trial Frequency & Lap Time ($T_{	ext{trial}}$)
### Theory
Measures the mean duration of complete bidirectional traversals (ida y vuelta) across the linear track by detecting upward zero-crossings or directional state transitions in the animal's movement vector `mov_dir`, yielding the fundamental behavioral frequency $f_{	ext{trial}} = 1 / T_{	ext{trial}}$.

In [ ]:
def compute_lap_time_synthetic(mov_dir, time_vec):
    dir_bin = (mov_dir > 0).astype(int)
    transitions = np.where(np.diff(dir_bin) == 1)[0] + 1
    if len(transitions) < 2:
        return np.nan, np.nan
    cycle_durations = np.diff(time_vec[transitions])
    valid = cycle_durations[(cycle_durations > 1.0) & (cycle_durations < 300.0)]
    if len(valid) == 0:
        return np.nan, np.nan
    mean_lap = float(np.mean(valid))
    return mean_lap, 1.0 / mean_lap

t_synth = np.linspace(0, 200, 4000)
mov_synth = np.sin(2 * np.pi * (1.0 / 15.0) * t_synth)
lap_t, lap_f = compute_lap_time_synthetic(mov_synth, t_synth)
print(f'Synthetic Lap Time Analysis -> Mean Lap Time: {lap_t:.2f}s | Trial Frequency: {lap_f:.4f} Hz')